# Feature Engineering para el dataset de Ames Housing

Este notebook ha sido generado para procesar `train.csv` con transformaciones adaptadas al perfil EDA.

---
## 1. Carga de datos y separación train/test

Dividiremos primero en train/test (80/20), luego definiremos el pipeline en train y lo aplicaremos a test. Toda ingeniería será sin data leakage.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Carga
data = pd.read_csv('data/train.csv')

# Target
TARGET = 'SalePrice'

# Separar features y target
y = data[TARGET]
X = data.drop(TARGET, axis=1)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0)

X_train.reset_index(drop=True, inplace=True)
X_test.reset_index(drop=True, inplace=True)
y_train.reset_index(drop=True, inplace=True)
y_test.reset_index(drop=True, inplace=True)

---
## 2. Imputación de variables categóricas
- Si >5% missing: imputar 'Missing'
- Si <5%: imputar moda

In [ ]:
categorical = X_train.select_dtypes(include='object').columns.tolist()
missing_pct = X_train[categorical].isnull().mean()

cat_missing_high = missing_pct[missing_pct > 0.05].index.tolist()
cat_missing_low = missing_pct[(missing_pct > 0) & (missing_pct <= 0.05)].index.tolist()

# Imputaciones
for col in cat_missing_high:
    fill_val = 'Missing'
    X_train[col] = X_train[col].fillna(fill_val)
    X_test[col] = X_test[col].fillna(fill_val)

for col in cat_missing_low:
    fill_val = X_train[col].mode()[0]
    X_train[col] = X_train[col].fillna(fill_val)
    X_test[col] = X_test[col].fillna(fill_val)

---
## 3. Imputación de variables numéricas
- Imputar media
- Crear indicador de missing

In [ ]:
numerical = X_train.select_dtypes(include=['number']).columns.tolist()
# Quitamos el target y 'Id'
numerical = [c for c in numerical if c not in ['SalePrice', 'Id']]
missing_num = X_train[numerical].isnull().sum()
missing_num_cols = missing_num[missing_num > 0].index.tolist()

for col in missing_num_cols:
    mean_val = X_train[col].mean()
    # Imputar
    X_train[col] = X_train[col].fillna(mean_val)
    X_test[col] = X_test[col].fillna(mean_val)
    # Indicador de missingness
    indicator = f'{col}_was_missing'
    X_train[indicator] = (X_train[col].isnull()).astype(int)
    X_test[indicator] = (X_test[col].isnull()).astype(int)

---
## 4. Transformación de variables temporales (años)
- Transformamos: 'YearBuilt', 'YearRemodAdd', 'GarageYrBlt'
- Restamos a 'YrSold' para crear años transcurridos

In [ ]:
time_cols = ['YearBuilt', 'YearRemodAdd', 'GarageYrBlt']
ref_year_col = 'YrSold'
for col in time_cols:
    new_col = col + '_since'
    # Nota: algunas filas pueden quedar negativas por inconsistencias
    X_train[new_col] = X_train[ref_year_col] - X_train[col]
    X_test[new_col] = X_test[ref_year_col] - X_test[col]

---
## 5. Transformación de variables numéricas altamente sesgadas (skew > 0.75)
- Se aplica log1p. Si hay valores negativos, aplicar Yeo-Johnson.

In [ ]:
from scipy.stats import yeojohnson

dist_columns = ['MiscVal', 'PoolArea', 'LotArea', '3SsnPorch', 'LowQualFinSF', 'KitchenAbvGr',
 'BsmtFinSF2', 'ScreenPorch', 'BsmtHalfBath', 'EnclosedPorch', 'MasVnrArea',
 'OpenPorchSF', 'LotFrontage', 'SalePrice', 'BsmtFinSF1', 'WoodDeckSF',
 'TotalBsmtSF', 'MSSubClass', '1stFlrSF', 'GrLivArea', 'BsmtUnfSF', '2ndFlrSF']

from sklearn.preprocessing import PowerTransformer

def safe_log1p(col):
    train_min = X_train[col].min()
    # Usamos log1p si la variable no tiene valores negativos
    if train_min >= 0:
        X_train[col] = np.log1p(X_train[col])
        X_test[col] = np.log1p(X_test[col])
    else:
        # fallback: Yeo-Johnson
        yj = PowerTransformer(method='yeo-johnson')
        X_train[col] = yj.fit_transform(X_train[[col]])
        X_test[col] = yj.transform(X_test[[col]])
for col in dist_columns:
    if col in X_train.columns:
        safe_log1p(col)

---
## 6. Variables con >50% ceros: binarización además del valor original

In [ ]:
candid_bin = ['MasVnrArea','BsmtFinSF2','2ndFlrSF','LowQualFinSF','BsmtFullBath','BsmtHalfBath','HalfBath','WoodDeckSF','EnclosedPorch','3SsnPorch','ScreenPorch','PoolArea','MiscVal']
for col in candid_bin:
    if col in X_train.columns:
        bin_name = col + '_iszero'
        X_train[bin_name] = (X_train[col] == 0).astype(int)
        X_test[bin_name] = (X_test[col] == 0).astype(int)

---
## 7. Encoding de variables categóricas especiales
- Ordinales conocidas (ej: 'ExterQual') con jerarquía basada en target
- Ordinales predefinidas: 'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Functional'

In [ ]:
ordinal_maps = {
    'ExterQual':   {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1, 'Missing':0},
    'ExterCond':   {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1, 'Missing':0},
    'BsmtQual':    {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1, 'Missing':0},
    'BsmtCond':    {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1, 'Missing':0},
    'HeatingQC':   {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1, 'Missing':0},
    'KitchenQual': {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1, 'Missing':0},
    'FireplaceQu': {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1, 'Missing':0},
    'GarageQual':  {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1, 'Missing':0},
    'GarageCond':  {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1, 'Missing':0},
    'PoolQC':      {'Ex':4, 'Gd':3, 'Fa':2, 'Po':1, 'Missing':0},
    'BsmtExposure':{'Gd':4, 'Av':3, 'Mn':2, 'No':1, 'Missing':0},
    'BsmtFinType1':{'GLQ':6,'ALQ':5,'BLQ':4,'Rec':3,'LwQ':2,'Unf':1, 'Missing':0},
    'BsmtFinType2':{'GLQ':6,'ALQ':5,'BLQ':4,'Rec':3,'LwQ':2,'Unf':1, 'Missing':0},
    'Functional':  {'Typ':7, 'Min1':6, 'Min2':5, 'Mod':4, 'Maj1':3, 'Maj2':2, 'Sev':1, 'Sal':0, 'Missing':-1}
}

for col, mapping in ordinal_maps.items():
    if col in X_train.columns:
        X_train[col] = X_train[col].map(mapping)
        X_test[col]  = X_test[col].map(mapping)
        # Imputar si quedan NAs tras el map
        X_train[col] = X_train[col].fillna(0)
        X_test[col] = X_test[col].fillna(0)

---
## 8. RareLabel Encoder y OrdinalEncoder para categóricas nominales de alta cardinalidad
- Para variables como 'Neighborhood', 'Exterior1st', 'Exterior2nd'

In [ ]:
HIGH_CARD_THRESHOLD = 10
cat_card = {col: X_train[col].nunique() for col in categorical if col in X_train.columns}
high_cardinality = [col for col,c in cat_card.items() if c > HIGH_CARD_THRESHOLD]

# RareLabelEncoder: etiquetas con frecuencia <1% → 'Rare'
def rare_label(col):
    freq = X_train[col].value_counts(normalize=True)
    rare = freq[freq < 0.01].index
    X_train[col] = X_train[col].replace(rare, 'Rare')
    X_test[col] = X_test[col].replace(rare, 'Rare')

for col in high_cardinality:
    rare_label(col)

# Ordinal encoding basado en la mediana del target (solo en train)
for col in high_cardinality:
    mapping = y_train.groupby(X_train[col]).median().sort_values().reset_index()
    mapping['code'] = range(1, mapping.shape[0]+1)
    map_dict = dict(zip(mapping.iloc[:,0], mapping['code']))
    X_train[col] = X_train[col].map(map_dict)
    X_test[col] = X_test[col].map(map_dict)
    # Imputar Rare o nuevas etiquetas no vistas con 0
    X_train[col] = X_train[col].fillna(0)
    X_test[col] = X_test[col].fillna(0)

---
## 9. Encoding ordinal estándar para el resto de variables categóricas
- Mapping igual usando mediana del target

In [ ]:
other_categoricals = [col for col in categorical if col not in ordinal_maps and col not in high_cardinality]
for col in other_categoricals:
    mapping = y_train.groupby(X_train[col]).median().sort_values().reset_index()
    mapping['code'] = range(1, mapping.shape[0]+1)
    map_dict = dict(zip(mapping.iloc[:,0], mapping['code']))
    X_train[col] = X_train[col].map(map_dict)
    X_test[col] = X_test[col].map(map_dict)
    X_train[col] = X_train[col].fillna(0)
    X_test[col] = X_test[col].fillna(0)

---
## 10. Escalado MinMax

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

---
## 11. Verificación final de NaNs e infinitos

In [ ]:
assert not X_train.isnull().any().any(), 'NaNs en X_train'
assert not X_test.isnull().any().any(), 'NaNs en X_test'
assert np.isfinite(X_train.to_numpy()).all(), 'Inf en X_train'
assert np.isfinite(X_test.to_numpy()).all(), 'Inf en X_test'
print("Transformaciones completadas correctamente")

---
## 12. Guardar objetos transformados para downstream (opcional)

In [ ]:
# Descomentar si se desea guardar
data_out = {'X_train': X_train, 'X_test': X_test, 'y_train': y_train, 'y_test': y_test}
# pd.to_pickle(data_out, 'data/features.pkl')